## Imports

In [ ]:
import numpy as np
import pandas as pd
import requests
from google.colab import drive
import matplotlib.pyplot as plt

## Data loading

In [ ]:
def data_loading(file_path, separator):
    drive.mount("/content/drive", force_remount=True)
    df = pd.read_csv(file_path, sep=separator)
    #list[tuple(riga parole csv)]
    return [tuple(row) for row in df.values.tolist()]

def data_cleaning(word_tuples):
    #list[tuple(riga parole csv non vuote o NaN)]
    return [word_tuple for word_tuple in word_tuples if word_tuple is not None and len(word_tuple) == total_langs and all(pd.notna(x) and str(x).strip() != "" for x in word_tuple)]

## Obtain lemma from synsetID

In [ ]:
def lemma_from_synsetID(synsets, synsetID, lang):
    for synset in synsets:
        properties = synset.get('properties') #root
        id_to_check = properties.get('synsetID').get('id')
        language = properties.get('language').upper()
        if id_to_check == synsetID and language == lang:
            if properties.get('fullLemma'):
                return properties.get('fullLemma')
            return properties.get('simpleLemma')
    return f"No lemma associated with id: {synsetID}"

## Information extraction from synset

In [ ]:
#extract for each language the corresponding synsetIDs from the response obtained vai BabelNet
def information_extraction_from_synset(synsets):
    processed_synsets = {} #dict(lingua, set(synsets per quella lingua))
    for synset in synsets:
        properties = synset.get('properties')
        synsetID = properties.get('synsetID').get('id')
        language = properties.get('language').upper()
        #print(f"props: {properties}, synset_id: {synsetID}, lang: {language}")
        if synsetID and language:
            if not language in processed_synsets: processed_synsets[language] = set()
            processed_synsets[language].add(synsetID)
    for lang in processed_synsets:
        print(f"Language: {lang}\nSynsets: {processed_synsets[lang]}")
    return processed_synsets

## Getting synsets from lemmas

In [ ]:
#obtaining the full json associated to the given lemma, containing all the info for each specified target language
def get_synsets(lemma, targetLang, key, source = "WIKI"):
    url = 'https://babelnet.io/v9/getSenses'
    searchLang = targetLang[0]
    params = {
        'lemma': lemma,
        'searchLang': searchLang,
        'targetLang': targetLang,
        'key': key,
        'source': source
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        #list[dict(proprietà babelnet lemma, valore associato)]
        return response.json()
    except Exception as e:
        print(f"An error has occurred: {e}")
        return None

## Ambiguity evaluation
$$
\text{AmbiguityReduction} = \frac{\sum_{i=1}^N |\mathcal{S}_i| - N \cdot \left|\bigcap_{i=1}^N \mathcal{S}_i\right|}{\sum_{i=1}^N |\mathcal{S}_i|}
$$
where:
- $ |\bigcap_{i=1}^N \mathcal{S}_i| \cdot N $ are the senses shared amongst languages.  
- $ \sum_{i=1}^N |\mathcal{S}_i| $ is the totality of senses amongst the languages.



In [ ]:
def ambiguity_evaluation(words, langs, api_key):
    if len(words) != len(langs):
        print("Length mismatch between words and languages!")
        return None

    synsets = get_synsets(words[0], langs, api_key)
    if not synsets:
        print(f"No synsets found for words: {words}")
        return None

    print("-------------------------------------------------------------------------------------\n")
    print(f"Words: {words}")
    processed_synsets = information_extraction_from_synset(synsets)
    synsets_sets = [processed_synsets.get(lang) for lang in langs]

    if not all(synsets_sets):
        print(f"AT LEAST ONE SYNSET FOUND EMPTY!")


    if not all(synsets_sets):
        synset_intersection = set()
    else:
        synset_intersection = synsets_sets[0]
        for current_synset in synsets_sets[1:]:
            synset_intersection = synset_intersection.intersection(current_synset)

    synset_intersection_length = len(synset_intersection)
    number_of_languages = len(langs)
    total_length = sum(len(synsets) if synsets else 0 for synsets in synsets_sets)

    #print(f"total_length: {total_length}, number_of_languages: {number_of_languages}, intersection_length:{intersection_length}")
    ambiguity_reduction = (total_length - (number_of_languages * synset_intersection_length)) / total_length if total_length != 0 else 0.0

    pseudoword = '-'.join(words)
    print("\033[0;32m\n-------------------\n|SYNSETS TO LEMMAS|\n-------------------\n\033[0m")
    print_synset_to_lemma(pseudoword, words, synsets, synset_intersection)

    return {'pseudoword': pseudoword,'ambiguity_reduction': f'{ambiguity_reduction:.3}'}

In [ ]:
def print_synset_to_lemma(pseudoword, words, synsets, synset_intersection):
    print(f"Synsets intersection: {synset_intersection}")
    for synsetID in synset_intersection:
      print(f"  Pseudoword: {pseudoword}, words: {words}")
      for language in langs:
        lemma = lemma_from_synsetID(synsets, synsetID, language)
        print(f"    language: {language}, sysnsetID: {synsetID}, lemma: {lemma}")
      print()

## Main

**Setup**

In [ ]:
api_key = "..."

total_langs = 4

#path = "/content/drive/MyDrive/.../6.Progetto"
path = "/content/drive/MyDrive/Uni/aa2425/TLN/DiCaro/6.Progetto"

match total_langs:
  case 2:
    langs = ["EN","IT"]
    input_file = f"{path}/prova_2.tsv"
  case 3:
    langs = ["EN","IT","FR"]
    input_file = f"{path}/prova_3.tsv"
  case 4:
    langs = ["EN","IT","ES","RO"]
    input_file = f"{path}/prova_4_1.tsv"

    #langs = ["EN","IT","FR","ES"]
    #input_file = f"{path}/prova_4.tsv"
  case 5:
    langs = ["EN","IT","FR","ES","RO"]
    input_file = f"{path}/prova_5_1.tsv"

    #langs = ["EN","IT","FR","ES","PT"]
    #input_file = f"{path}/prova_5.tsv"
  case 6:
    langs = ["EN","IT","FR","ES","PT","RO"]
    input_file = f"{path}/Test.tsv"

**Main**

In [ ]:
data = data_loading(input_file, separator="\t")
clean_tuples = data_cleaning(data)
ambiguity_scores = []
result = dict()

for idx,words in enumerate(clean_tuples):
  processed = ambiguity_evaluation(words, langs=langs, api_key=api_key)
  result[processed['pseudoword']] = float(processed['ambiguity_reduction'])

if result:
  print("\033[0;36m\n---------\n|RESULTS|\n---------\n\033[0m")
  result = dict(sorted(result.items(), key=lambda result: result[1], reverse=True))
  pseudowords = list(result.keys())
  ambiguities = list(result.values())
  for idx, pseudoword in enumerate(pseudowords):
    print(f"Pseudoword: {pseudoword}, ambiguity_reduction: {ambiguities[idx]}")

Mounted at /content/drive
-------------------------------------------------------------------------------------

Words: ('water', 'acqua', 'agua', 'apă')
Language: ES
Synsets: {'bn:00534958n', 'bn:00042379n', 'bn:24545895n', 'bn:08389865n', 'bn:01035397n', 'bn:01142092n', 'bn:21931991n', 'bn:00028760n', 'bn:00061253n', 'bn:03316593n', 'bn:00080561n', 'bn:17524168n', 'bn:00257725n', 'bn:00080562n', 'bn:14976572n', 'bn:01479284n', 'bn:00080637n', 'bn:00011766n', 'bn:02226053n', 'bn:02007412n', 'bn:17366259n', 'bn:00045705n', 'bn:03875209n'}
Language: EN
Synsets: {'bn:03628978n', 'bn:21129700n', 'bn:25057743n', 'bn:02047388n', 'bn:00042379n', 'bn:03081532n', 'bn:01484615n', 'bn:01410923n', 'bn:01553698n', 'bn:00639160n', 'bn:24545895n', 'bn:22217043n', 'bn:30496356n', 'bn:23535911n', 'bn:01035397n', 'bn:01142092n', 'bn:21931991n', 'bn:21114671n', 'bn:17650628n', 'bn:00028760n', 'bn:00061253n', 'bn:29555302n', 'bn:03316593n', 'bn:00168780n', 'bn:00080561n', 'bn:02123811n', 'bn:24136471n', 